# Experiment: IFEval Instruction Selector Playground

Goal:
- Tune prompt/setup for an external selector LLM that picks which instruction ids to boost next.
- Validate JSON robustness and confidence behavior before integrating into dynamic boosting.


## How to use

1. Pick a sample and edit `STEP_GENERATIONS` if needed.
2. Tweak `SYSTEM_PROMPT` and/or `build_selector_payload`.
3. Run the evaluation loop and inspect decisions.
4. Keep only prompt variants that stay JSON-valid and semantically reasonable.


In [ ]:
from __future__ import annotations

import json
import os
import re
import urllib.error
import urllib.request
from pprint import pprint


In [ ]:
# Selector backend config
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
OLLAMA_CHAT_URL = OLLAMA_BASE_URL.rstrip("/") + "/api/chat"
SELECTOR_MODEL = "gpt-oss:20b"

print("Ollama endpoint:", OLLAMA_CHAT_URL)
print("Selector model:", SELECTOR_MODEL)


In [ ]:
# Minimal IFEval-like sample (instruction-last format)
SAMPLE = {
    "sample_id": "demo_ifeval_001",
    "base_question": "Write a short answer about why exercise is useful.",
    "instruction_id_list": [
        "length:at_most_3_sentences",
        "style:must_include_word_because",
        "format:bullet_list",
    ],
    "instruction_texts": [
        "Use at most 3 sentences.",
        "Include the word 'because'.",
        "Format the answer as a bullet list.",
    ],
}

STEP_GENERATIONS = [
    "Exercise helps with mood and energy.",
    "It is useful because it also supports better sleep.",
    "- Exercise improves focus and mood because it reduces stress.",
    "- It also supports sleep and energy levels.",
]

pprint(SAMPLE)


In [ ]:
SYSTEM_PROMPT = """You are an instruction-selector for dynamic attention boosting.\nReturn ONLY valid JSON with keys: decision, active_instruction_ids, confidence, reason.\ndecision must be one of: stay, switch, add.\nactive_instruction_ids must be a non-empty list of ids from candidate_instruction_ids.\nconfidence must be a float in [0,1].\nKeep reason short."""

def build_selector_payload(sample: dict, current_generation: str, active_ids: list[str]) -> dict:
    return {
        "sample_id": sample["sample_id"],
        "base_question": sample["base_question"],
        "candidate_instruction_ids": sample["instruction_id_list"],
        "instruction_texts": [
            {"id": i, "text": t}
            for i, t in zip(sample["instruction_id_list"], sample["instruction_texts"])
        ],
        "currently_active_instruction_ids": active_ids,
        "current_generation": current_generation,
    }

def extract_json_object(raw_text: str) -> dict:
    raw_text = raw_text.strip()
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{[\s\S]*\}", raw_text)
    if not match:
        raise ValueError("No JSON object found in selector output")
    return json.loads(match.group(0))

def validate_selector_output(obj: dict, candidate_ids: list[str]) -> dict:
    if obj.get("decision") not in {"stay", "switch", "add"}:
        raise ValueError(f"Invalid decision: {obj.get('decision')}")

    selected = obj.get("active_instruction_ids")
    if not isinstance(selected, list) or not selected:
        raise ValueError("active_instruction_ids must be a non-empty list")

    for inst_id in selected:
        if inst_id not in candidate_ids:
            raise ValueError(f"Unknown instruction id in selection: {inst_id}")

    confidence = obj.get("confidence")
    if not isinstance(confidence, (int, float)):
        raise ValueError("confidence must be numeric")
    confidence = max(0.0, min(1.0, float(confidence)))

    reason = obj.get("reason", "")
    if not isinstance(reason, str):
        reason = str(reason)

    return {
        "decision": obj["decision"],
        "active_instruction_ids": selected,
        "confidence": confidence,
        "reason": reason.strip(),
    }


In [ ]:
def ollama_chat(model: str, system_prompt: str, user_payload: dict, temperature: float = 0.0) -> str:
    body = {
        "model": model,
        "stream": False,
        "options": {"temperature": temperature},
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": json.dumps(user_payload, ensure_ascii=True, indent=2)},
        ],
    }
    req = urllib.request.Request(
        OLLAMA_CHAT_URL,
        data=json.dumps(body).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            payload = json.loads(resp.read().decode("utf-8"))
    except urllib.error.URLError as exc:
        raise RuntimeError(f"Failed to call Ollama at {OLLAMA_CHAT_URL}: {exc}") from exc

    return payload["message"]["content"]

def fallback_selector(sample: dict, active_ids: list[str]) -> dict:
    # Conservative fallback: keep current if any, otherwise start with first candidate.
    chosen = active_ids[:] if active_ids else [sample["instruction_id_list"][0]]
    return {
        "decision": "stay",
        "active_instruction_ids": chosen,
        "confidence": 0.25,
        "reason": "Fallback selector",
    }

def select_instructions(sample: dict, current_generation: str, active_ids: list[str]) -> dict:
    payload = build_selector_payload(sample, current_generation, active_ids)
    raw = ollama_chat(SELECTOR_MODEL, SYSTEM_PROMPT, payload, temperature=0.0)
    try:
        parsed = extract_json_object(raw)
        return validate_selector_output(parsed, sample["instruction_id_list"])
    except Exception:
        return fallback_selector(sample, active_ids)


In [ ]:
# Dry-run parser tests before hitting the model
GOOD = '{"decision":"switch","active_instruction_ids":["style:must_include_word_because"],"confidence":0.84,"reason":"Now satisfy because-word rule"}'
MESSY = 'Result:\n```json\n{"decision":"add","active_instruction_ids":["format:bullet_list","style:must_include_word_because"],"confidence":0.73,"reason":"Need bullet formatting and keyword"}\n```'

print(validate_selector_output(extract_json_object(GOOD), SAMPLE["instruction_id_list"]))
print(validate_selector_output(extract_json_object(MESSY), SAMPLE["instruction_id_list"]))


In [ ]:
# Stepwise selector simulation
history = []
active = []
current_text = ""

for i, sent in enumerate(STEP_GENERATIONS, start=1):
    current_text = (current_text + " " + sent).strip()
    decision = select_instructions(SAMPLE, current_text, active)
    active = decision["active_instruction_ids"]

    row = {
        "step": i,
        "sentence": sent,
        "decision": decision["decision"],
        "active_instruction_ids": decision["active_instruction_ids"],
        "confidence": decision["confidence"],
        "reason": decision["reason"],
    }
    history.append(row)

    print(f"Step {i}: {row['decision']} | active={row['active_instruction_ids']} | conf={row['confidence']:.2f}")

history


In [ ]:
# Quick summary
avg_conf = sum(r["confidence"] for r in history) / len(history) if history else 0.0
print(f"Steps: {len(history)}")
print(f"Average confidence: {avg_conf:.3f}")
print(f"Final active instruction ids: {history[-1]['active_instruction_ids'] if history else []}")


## Notes

- Keep this notebook focused on selector prompt quality and parse stability.
- Dynamic boost integration belongs in benchmark scripts/modules, not in this notebook.
- When prompt changes are adopted, copy final system prompt text into versioned code under `src/dynamic_boost/selector_llm.py`.


## Synthetic Booster Switching Demo

This demo builds a tiny synthetic attention module and runs the dynamic controller with a programmed selector:
- Step 1: boost `i1`
- Boundary 1: switch to `i2`
- Boundary 2: add `i3`

It then plots per-step attention distribution over three token positions to verify the booster is updating correctly.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import sys
from pathlib import Path

# Ensure project root is importable when notebook runs from notebooks/
_cwd = Path.cwd().resolve()
_repo = _cwd if (_cwd / 'src').exists() else _cwd.parent
if not (_repo / 'src').exists():
    raise RuntimeError(f'Could not locate project root from cwd={_cwd}')
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))

from src import register_boost_hooks, unregister_boost_hooks, update_bias_mask
from src.dynamic_boost import BoundaryChecker, BoundaryConfig, DynamicBoostController, TokenStepOutput
from src.dynamic_boost.types import SelectorDecision, SelectorRequest
from src.ifeval_dynamic.runner import build_active_boost_config

class _DummyAttn(nn.Module):
    def forward(self, scores: torch.Tensor) -> torch.Tensor:
        return F.softmax(scores, dim=-1)

class _DummyModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.block = nn.Module()
        self.block.attn = _DummyAttn()
        self.anchor = nn.Parameter(torch.zeros(1))

class _ProgrammedSelector:
    def __init__(self) -> None:
        self.calls = 0

    def select(self, _request: SelectorRequest) -> SelectorDecision:
        self.calls += 1
        if self.calls == 1:
            return SelectorDecision(
                decision='switch',
                active_instruction_ids=['i2'],
                confidence=1.0,
                reason='switch_to_i2',
            )
        return SelectorDecision(
            decision='add',
            active_instruction_ids=['i3'],
            confidence=1.0,
            reason='add_i3',
        )

model = _DummyModel()
instruction_token_indices = {'i1': [0], 'i2': [1], 'i3': [2]}
boost_bias = 6.0
initial_active = ['i1']

cfg = build_active_boost_config(instruction_token_indices, initial_active, boost_bias=boost_bias)
handle = register_boost_hooks(model, cfg, input_length=3)
update_bias_mask(handle, seq_length=3, device=torch.device('cpu'))

selector = _ProgrammedSelector()
checker = BoundaryChecker(BoundaryConfig(min_tokens_between_checks=1, max_tokens_without_check=99, boundary_markers=('.',), rolling_buffer_chars=16))

attn_steps = []

def step_fn(_active_ids, step_index):
    scores = torch.tensor([[0.0, 0.0, 0.0]], dtype=torch.float32)
    attn = model.block.attn(scores).detach().clone()[0]
    attn_steps.append(attn)
    if step_index <= 2:
        return TokenStepOutput(text='x.')
    return TokenStepOutput(text='x', is_eos=True)

def request_builder(current_generation, active_ids, generation_token_count, step_index):
    return SelectorRequest(
        sample_id='synthetic_1',
        base_prompt='q',
        candidate_instruction_ids=['i1', 'i2', 'i3'],
        instruction_text_by_id={'i1': 'A', 'i2': 'B', 'i3': 'C'},
        currently_active_instruction_ids=active_ids,
        current_generation=current_generation,
        generation_token_count=generation_token_count,
        step_index=step_index,
        metadata={},
    )

def on_selector_update(next_active_ids, _decision, _event):
    handle.config = build_active_boost_config(instruction_token_indices, next_active_ids, boost_bias=boost_bias)
    update_bias_mask(handle, seq_length=3, device=torch.device('cpu'))

controller = DynamicBoostController(
    model_name='dummy',
    selector_backend='synthetic',
    selector=selector,
    boundary_checker=checker,
    step_fn=step_fn,
    request_builder=request_builder,
    on_selector_update=on_selector_update,
    decode_config={'max_new_tokens': 3},
)

try:
    result = controller.run(
        sample_id='synthetic_1',
        initial_active_instruction_ids=initial_active,
        max_new_tokens=3,
    )
finally:
    unregister_boost_hooks(handle)

print('Final active ids:', result.final_active_instruction_ids)
print('Selector calls:', result.trace.selector_calls)
print('Boundary events:', [e.reason for e in result.trace.boundary_events])
print('Decisions:', [(d.decision, d.active_instruction_ids, d.reason) for d in result.trace.selector_decisions])

x = [0, 1, 2]
labels = ['token0(i1)', 'token1(i2)', 'token2(i3)']
fig, axes = plt.subplots(1, len(attn_steps), figsize=(12, 3), sharey=True)
if len(attn_steps) == 1:
    axes = [axes]
for i, (ax, attn) in enumerate(zip(axes, attn_steps), start=1):
    ax.bar(x, attn.numpy())
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha='right')
    ax.set_title(f'Step {i}')
    ax.set_ylim(0.0, 1.0)
axes[0].set_ylabel('attention')
fig.suptitle('Synthetic Dynamic Boost: Attention Shift Across Steps')
fig.tight_layout()
plt.show()
